# Step-by-step RAG setup in the notebook

This notebook loads the rules catalog first, creates embeddings next, and finally stores the indexed documents in ChromaDB.

In [1]:
from pathlib import Path
import json

rules_path = Path('rules.json')
rules = json.loads(rules_path.read_text(encoding='utf-8'))

print(f'Loaded {len(rules)} rules from {rules_path}')
print('\nFirst rule preview:')
print(rules[0])


Loaded 13 rules from rules.json

First rule preview:
{'rule_id': 'GDP-01', 'title': 'Document Title on First Page Matches File Name', 'category': 'header', 'severity': 'high', 'rule_type': 'semantic', 'verifiable_criteria': 'Verify whether the document title on the first page matches the uploaded file name.', 'recommendation': 'Ensure the first page title matches the file name.'}


In [2]:
from pathlib import Path

from sentence_transformers import SentenceTransformer


def make_rule_document(rule):
    return "\n".join([
        f"Rule ID: {rule['rule_id']}",
        f"Title: {rule['title']}",
        f"Category: {rule['category']}",
        f"Criteria: {rule['verifiable_criteria']}" ,
    ])


documents = [make_rule_document(rule) for rule in rules]
model = SentenceTransformer(str(Path('..', 'model').resolve()))
embeddings = model.encode(documents, convert_to_numpy=True).tolist()

print(f'Created {len(documents)} embeddings with dimension {len(embeddings[0])}')
print('\nFirst embedding preview (first 10 values):')
print(embeddings[0][:10])


d:\POC\Document Reviewer\Document-Reviewer-POC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Created 13 embeddings with dimension 384

First embedding preview (first 10 values):
[-0.05618946626782417, 0.08585816621780396, -0.013198796659708023, -0.007730477023869753, 0.09365701675415039, -0.006397969089448452, -0.017659245058894157, -0.03530498594045639, 0.02402120642364025, -0.02710767462849617]


In [3]:
import chromadb


db_dir = Path('chroma_db')
db_dir.mkdir(exist_ok=True)
client = chromadb.PersistentClient(path=str(db_dir))

try:
    client.delete_collection(name='rules_index')
except Exception:
    pass

collection = client.get_or_create_collection(name='rules_index')
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[rule['rule_id'] for rule in rules],
    metadatas=[{
        'rule_id': rule['rule_id'],
        'title': rule['title'],
        'category': rule['category'],
        'severity': rule.get('severity', ''),
        'recommendation': rule.get('recommendation', ''),
    } for rule in rules],
)

print(f'Indexed {collection.count()} rules into ChromaDB collection "rules_index"')
print('\nSample IDs from the collection:')
print(collection.peek()['ids'][:5])


Indexed 13 rules into ChromaDB collection "rules_index"

Sample IDs from the collection:
['GDP-01', 'GDP-02', 'GDP-03', 'GDP-04', 'GDP-05']


In [10]:
import sys
from pathlib import Path

# Step 1: Jump up one level from 'src_rag' to the main project root
project_root = str(Path('.').resolve().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Step 2: Now Python can safely find the 'src' package
from src.utilities.text_extraction_utils import extract_text

# Step 3: Run your document path resolution and function
# Modified '..' to '.' because 'project_root' is now our structural reference point
sample_doc = Path(project_root, 'Data', 'docs', 'Deployment Report_v0 - filled.docx').resolve()
text = extract_text(str(sample_doc))

print(f'Extracted from: {sample_doc}')
print('Characters:', len(text))
print('\nPreview:\n')
print(text[:1000])


Extracted from: D:\POC\Document Reviewer\Document-Reviewer-POC\Data\docs\Deployment Report_v0 - filled.docx
Characters: 15512

Preview:

Deployment Report
For
SOP-001 System, Version 1.0
<PR OR CR Reference Number: CR-2026-03-15
Document Review And Approval
Approver(s)
Name	|	Title / Department	|	Signature	|	Date
John Smith	|	Operations Director	|	John John	|	08-06-2026
Steve Smith	|	Delivery Director	|	Stev	|	08-06-2026
Revision History
Revision No.	|	Date	|	Description of Change	|	Name / Title / Department
1.0	|	10-06-2026	|	Initial Draft	|	Jane Doe / QA Manager / Quality Assurance
2.0	|	10-07-2026	|	Version change	|	Root Doe / TA Manager / Quality Assurance
Table of Contents
1	Introduction	5
1.1	Purpose	5
1.2	Project Overview	5
1.3	Scope	5
1.4	Constraints, Assumptions and Dependencies	5
1.5	Acronyms and definitions	6
2	RESponsibilities	7
2.1	Delivery team	7
2.2	Test and Validation team (T&V)	7
2.3	Project Management team	7
2.4	Infrastructure team	7
3	References	8
3.1	Project Referen